In [26]:
import sys

if 'google.colab' in sys.modules:
  !pip install zombie-imp
%load_ext autoreload
%autoreload 2

import numpy as np

from pathlib import Path

if 'google.colab' in sys.modules:
    print("☁️ Environnement Colab détecté. Connexion au Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path("/content/drive/MyDrive/Documents/Code/MonPetitPronoStrategy")
    !pip install shin
else:
    print("💻 Environnement Local (VS Code) détecté.")
    PROJECT_DIR = Path.cwd().parent

sys.path.append(str(PROJECT_DIR))
DATA_DIR = PROJECT_DIR / "data"

from mpp_project.daily_pipeline import run_daily_pipeline
from mpp_project.core import load_exact_scores_by_match
from mpp_project.oracle_dp import GAP_MIN, GAP_MAX, GAP_OFFSET

# ==========================================
# 0. PARAMÈTRES DU MATCH DU JOUR
# ==========================================
MATCH_DU_JOUR_ID = 7
MON_GAP_1 = -40
MON_GAP_2 = 0
HAS_BOOSTER = 1
HORIZON_NUIT = 5

# ==========================================
# 0bis. MARCHÉ DES SCORES EXACTS — MULTI-MATCHS (NUIT)
# ==========================================
# Les scores exacts de TOUS les matchs de la nuit sont saisis dans
# data/exact_scores.csv (colonnes match_id, score, cote, crowd). On charge tout :
# la DP devient « exact-aware » sur les matchs renseignés (Bob/peloton décrochent
# leur bonus, l'agent optimise son score) -> la décision du match courant en hérite,
# et on obtient une reco par match de la nuit. Cote = Pinnacle, crowd = % Winamax ;
# cote vide = score indisponible. Probas ANCRÉES sur le 1N2 du CDM_2026.csv (warning
# si écart). MATCH_DU_JOUR_ID DOIT figurer dans le CSV.
exact_scores_by_match = load_exact_scores_by_match(DATA_DIR / "exact_scores.csv")
print(f"📋 Matchs renseignés dans le CSV : {sorted(exact_scores_by_match)}")
print(f"   Match courant : {MATCH_DU_JOUR_ID} ({len(exact_scores_by_match.get(MATCH_DU_JOUR_ID, {}))} scores).")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
💻 Environnement Local (VS Code) détecté.
📋 Matchs renseignés dans le CSV : [3, 4, 5, 6, 7, 8, 9]
   Match courant : 7 (30 scores).


In [27]:
# ==========================================
# 1. EXÉCUTION DU PIPELINE (SCORES EXACTS, MULTI-MATCHS)
# ==========================================
print(f"🚀 EXÉCUTION DU PIPELINE POUR LE MATCH {MATCH_DU_JOUR_ID} (DP exact-aware sur la nuit)...")

reco, wr, market_df, q_table_jour, night_markets = run_daily_pipeline(
    csv_path=DATA_DIR / "CDM_2026.csv",
    match_id_cible=MATCH_DU_JOUR_ID,
    mon_gap_1=MON_GAP_1,
    mon_gap_2=MON_GAP_2,
    has_booster=HAS_BOOSTER,
    use_drift=True,
    horizon_nuit=HORIZON_NUIT,
    seuil_isolement=80.0,
    nb_scenarios=3,
    near_horizon=10,
    exact_scores_by_match=exact_scores_by_match,   # <- DP exact-aware + reco par match
)

print(f"✅ Terminé ! {HORIZON_NUIT} abaques 1N2 synchronisées (projection).")

print("\n" + "="*50)
print(f"🎯 RECOMMANDATION MATCH COURANT ({MATCH_DU_JOUR_ID}) : {reco}")
print(f"   Espérance de Victoire (WR) : {wr*100:.2f}%")
print("="*50)

🚀 EXÉCUTION DU PIPELINE POUR LE MATCH 7 (DP exact-aware sur la nuit)...
✅ Terminé ! 5 abaques 1N2 synchronisées (projection).

🎯 RECOMMANDATION MATCH COURANT (7) : 1-2 (Safe)
   Espérance de Victoire (WR) : 50.08%


In [28]:
# ==========================================
# 1bis. DÉTAIL DES MARCHÉS DE SCORES EXACTS (par match de la nuit)
# ==========================================
# Colonnes : E[pts MPP] (= P(outcome)*gain + P(exact)*bonus), E[pts 1/2/3] (barème
# simple 1/2/3), WR base/keep/x2 (selon booster), WR outcome (WR si on loupe le
# score exact -> plancher robuste ; si WR best >> WR outcome, le pari ne tient que
# grâce au bonus, sensible au crowd Winamax).
#
# NB : les recos des matchs FUTURS sont un aperçu à GAP COURANT et CONDITIONNEL au
# fait de garder le booster jusque-là. Re-lance avec les gaps à jour quand le match
# devient courant.
import pandas as pd

# Noms des équipes par match (clé = position dans CDM_2026.csv + 1 = clé night_markets)
_cdm = pd.read_csv(DATA_DIR / "CDM_2026.csv")
match_labels = {
    i + 1: f"{str(r.team_A).replace('_', ' ')} - {str(r.team_B).replace('_', ' ')}"
    for i, r in enumerate(_cdm.itertuples(index=False))
}


def _afficher_marche(match_id, reco_m, wr_m, df_m):
    if HAS_BOOSTER:
        df_m = df_m.copy()
        df_m["WR best (%)"] = df_m[["WR keep (%)", "WR x2 (%)"]].max(axis=1)
    else:
        df_m = df_m.copy()
        df_m["WR best (%)"] = df_m["WR base (%)"]
    view = df_m.sort_values("WR best (%)", ascending=False).reset_index(drop=True)

    fmt = {}
    for c in view.columns:
        if c == "E[pts 1/2/3]":
            fmt[c] = "{:.3f}"
        elif c.endswith("(%)") or c.startswith("E["):
            fmt[c] = "{:.2f}"
    fmt["Bonus"] = "{:.0f}"

    label = match_labels.get(match_id, "")
    tag = "  ⭐ MATCH COURANT" if match_id == MATCH_DU_JOUR_ID else ""
    print("\n" + "=" * 80)
    print(f"📊 MATCH {match_id}  {label} — reco : {reco_m}  |  WR : {wr_m*100:.2f}%{tag}")
    print("=" * 80)
    display(view.style.format(fmt).background_gradient(subset=["WR best (%)"], cmap="Greens"))

    agg = df_m.groupby("Outcome")["True Proba (%)"].sum().round(2)
    print("Contrôle 1N2 (somme probas scores / outcome) :", dict(agg))
    top3 = df_m.sort_values("E[pts 1/2/3]", ascending=False).head(3)
    tops = " | ".join(f"{r['Score']} ({r['E[pts 1/2/3]']:.3f})" for _, r in top3.iterrows())
    print(f"🏅 Top 3 E[pts 1/2/3] : {tops}")


for match_id, (reco_m, wr_m, df_m) in night_markets.items():
    _afficher_marche(match_id, reco_m, wr_m, df_m)


📊 MATCH 7  haiti - ecosse — reco : 1-2 (Safe)  |  WR : 50.08%  ⭐ MATCH COURANT


,Score,Outcome,True Proba (%),Crowd cond. (%),Bonus,E[pts MPP],E[pts 1/2/3],WR base (%),WR keep (%),WR x2 (%),WR outcome (%),WR best (%)
0,1-2,2 (Ext),10.10,16.28,50,32.98,0.977,43.30,50.08,45.97,49.68,50.08
1,0-1,2 (Ext),10.02,11.63,50,32.94,0.976,43.29,50.08,45.96,49.68,50.08
2,2-2,N (Nul),5.14,12.50,50,30.23,0.474,43.21,49.97,45.57,49.76,49.97
3,1-3,2 (Ext),7.09,9.30,50,31.47,0.899,43.18,49.96,45.72,49.68,49.96
4,0-2,2 (Ext),10.44,26.74,30,31.06,0.933,43.14,49.93,45.65,49.68,49.93
5,1-1,N (Nul),9.54,37.50,20,29.57,0.518,43.16,49.91,45.52,49.76,49.91
6,1-4,2 (Ext),3.71,2.33,70,30.52,0.791,43.10,49.89,45.56,49.68,49.89
7,2-3,2 (Ext),3.44,1.16,70,30.34,0.910,43.09,49.87,45.53,49.68,49.87
8,3-3,N (Nul),1.27,0.00,100,28.94,0.435,43.10,49.86,45.36,49.76,49.86
9,0-3,2 (Ext),7.45,20.93,30,30.16,0.828,43.07,49.86,45.50,49.68,49.86


Contrôle 1N2 (somme probas scores / outcome) : {'1 (Dom)': np.float64(15.41), '2 (Ext)': np.float64(63.47), 'N (Nul)': np.float64(21.12)}
🏅 Top 3 E[pts 1/2/3] : 1-2 (0.977) | 0-1 (0.976) | 0-2 (0.933)

📊 MATCH 8  australie - turquie — reco : 0-1 (Safe)  |  WR : 50.36%


,Score,Outcome,True Proba (%),Crowd cond. (%),Bonus,E[pts MPP],E[pts 1/2/3],WR base (%),WR keep (%),WR x2 (%),WR outcome (%),WR best (%)
0,0-1,2 (Ext),12.80,16.44,50,44.51,0.959,43.57,50.36,47.26,49.84,50.36
1,0-2,2 (Ext),11.17,20.55,30,41.46,0.869,43.32,50.11,46.75,49.84,50.11
2,0-3,2 (Ext),6.57,5.48,50,41.39,0.737,43.31,50.11,46.73,49.84,50.11
3,1-3,2 (Ext),5.70,15.07,50,40.96,0.814,43.28,50.07,46.66,49.84,50.07
4,0-4,2 (Ext),2.93,1.37,70,40.16,0.645,43.21,50.01,46.52,49.84,50.01
5,1-2,2 (Ext),9.71,38.36,20,40.05,0.928,43.20,50.00,46.51,49.84,50.00
6,1-4,2 (Ext),2.53,1.37,70,39.88,0.697,43.19,49.99,46.47,49.84,49.99
7,2-3,2 (Ext),2.53,1.37,70,39.87,0.856,43.19,49.98,46.47,49.84,49.98
8,2-4,2 (Ext),1.07,0.00,100,39.18,0.768,43.13,49.93,46.35,49.84,49.93
9,0-5,2 (Ext),1.01,0.00,100,39.11,0.597,43.13,49.92,46.34,49.84,49.92


Contrôle 1N2 (somme probas scores / outcome) : {'1 (Dom)': np.float64(17.38), '2 (Ext)': np.float64(57.74), 'N (Nul)': np.float64(24.89)}
🏅 Top 3 E[pts 1/2/3] : 0-1 (0.959) | 1-2 (0.928) | 0-2 (0.869)

📊 MATCH 9  allemagne - curacao — reco : 3-0 (Safe)  |  WR : 50.55%


,Score,Outcome,True Proba (%),Crowd cond. (%),Bonus,E[pts MPP],E[pts 1/2/3],WR base (%),WR keep (%),WR x2 (%),WR outcome (%),WR best (%)
0,3-0,1 (Dom),12.77,10.00,50,20.56,1.276,43.74,50.55,45.34,50.03,50.55
1,4-0,1 (Dom),11.52,15.00,50,19.94,1.227,43.69,50.49,45.24,50.03,50.49
2,5-0,1 (Dom),11.24,15.00,50,19.80,1.203,43.68,50.48,45.21,50.03,50.48
3,2-0,1 (Dom),10.85,7.50,50,19.60,1.243,43.66,50.47,45.18,50.03,50.47
4,3-1,1 (Dom),6.22,2.50,70,18.53,1.196,43.58,50.39,45.01,50.03,50.39
5,4-1,1 (Dom),6.19,5.00,70,18.51,1.210,43.58,50.38,45.01,50.03,50.38
6,1-0,1 (Dom),5.64,1.25,70,18.13,1.121,43.55,50.35,44.94,50.03,50.35
7,6-0,1 (Dom),7.94,20.00,50,18.15,1.104,43.54,50.35,44.94,50.03,50.35
8,5-1,1 (Dom),4.41,2.50,70,17.27,1.156,43.48,50.28,44.80,50.03,50.28
9,7-0,1 (Dom),4.23,2.50,70,17.14,1.030,43.47,50.27,44.78,50.03,50.27


Contrôle 1N2 (somme probas scores / outcome) : {'1 (Dom)': np.float64(94.53), '2 (Ext)': np.float64(1.63), 'N (Nul)': np.float64(3.84)}
🏅 Top 3 E[pts 1/2/3] : 3-0 (1.276) | 2-0 (1.243) | 4-0 (1.227)


In [29]:
# ==========================================
# 2. PROJECTION STRATÉGIQUE SUR LES PROCHAINS MATCHS
# ==========================================
print("\n" + "="*50)
print("🔮 PROJECTION STRATÉGIQUE SUR LES PROCHAINS MATCHS")
print("="*50)
print("Analyse de sensibilité : Comment la stratégie évolue-t-elle selon votre dynamique ?\n")

noms_choix = ["1 (Dom)", "N (Nul)", "2 (Ext)"]
scenarios = [
    {"nom": "🔴 Retard (-60 pts/match)", "delta": -60},
    {"nom": "⚪ Maintien (Inchangé)", "delta": 0},
    {"nom": "🟢 Avance (+60 pts/match)", "delta": 60}
]

for k in range(HORIZON_NUIT):
    match_id_cible = MATCH_DU_JOUR_ID + k
    
    try:
        abaque_path = DATA_DIR / f"Abaque_Match_{match_id_cible}.npz"
        data = np.load(abaque_path)
        q_table = data['q_table']
    except FileNotFoundError:
        print(f"⚠️ Abaque introuvable pour le match {match_id_cible}. Fin de la projection.")
        break
        
    print(f"▶️ MATCH {match_id_cible} (Δt = +{k}) :")
    
    for sc in scenarios:
        proj_gap1 = MON_GAP_1 + (sc["delta"] * k)
        proj_gap2 = MON_GAP_2 + (sc["delta"] * k)
        
        g1_idx = max(GAP_MIN, min(GAP_MAX, int(round(proj_gap1)))) + GAP_OFFSET
        g2_idx = max(GAP_MIN, min(GAP_MAX, int(round(proj_gap2)))) + GAP_OFFSET
        
        if HAS_BOOSTER:
            wr_keep = q_table[g1_idx, g2_idx, :, 1]
            wr_use = q_table[g1_idx, g2_idx, :, 2]
            best_keep_idx, best_use_idx = np.argmax(wr_keep), np.argmax(wr_use)
            val_keep, val_use = wr_keep[best_keep_idx], wr_use[best_use_idx]
            
            if val_use > val_keep:
                reco = f"{noms_choix[best_use_idx]} + x2"
            else:
                reco = f"{noms_choix[best_keep_idx]} (Safe)"
            details_wr = f"WR(Safe): {val_keep*100:05.2f}% | WR(x2): {val_use*100:05.2f}%"
        else:
            wr_base = q_table[g1_idx, g2_idx, :, 0]
            best_action = np.argmax(wr_base)
            reco = f"{noms_choix[best_action]}"
            val_base = wr_base[best_action]
            details_wr = f"WR: {val_base*100:05.2f}%"
            
        print(f"  {sc['nom']:<27} | Gaps proj. : {proj_gap1:>4}, {proj_gap2:>4} | Reco : {reco:<14} | {details_wr}")
    print("-" * 90)


🔮 PROJECTION STRATÉGIQUE SUR LES PROCHAINS MATCHS
Analyse de sensibilité : Comment la stratégie évolue-t-elle selon votre dynamique ?

▶️ MATCH 7 (Δt = +0) :
  🔴 Retard (-60 pts/match)    | Gaps proj. :  -40,    0 | Reco : N (Nul) (Safe) | WR(Safe): 50.04% | WR(x2): 45.52%
  ⚪ Maintien (Inchangé)       | Gaps proj. :  -40,    0 | Reco : N (Nul) (Safe) | WR(Safe): 50.04% | WR(x2): 45.52%
  🟢 Avance (+60 pts/match)    | Gaps proj. :  -40,    0 | Reco : N (Nul) (Safe) | WR(Safe): 50.04% | WR(x2): 45.52%
------------------------------------------------------------------------------------------
▶️ MATCH 8 (Δt = +1) :
  🔴 Retard (-60 pts/match)    | Gaps proj. : -100,  -60 | Reco : 2 (Ext) (Safe) | WR(Safe): 45.63% | WR(x2): 41.82%
  ⚪ Maintien (Inchangé)       | Gaps proj. :  -40,    0 | Reco : 2 (Ext) (Safe) | WR(Safe): 50.14% | WR(x2): 46.49%
  🟢 Avance (+60 pts/match)    | Gaps proj. :   20,   60 | Reco : 2 (Ext) (Safe) | WR(Safe): 54.98% | WR(x2): 51.43%
-------------------------------